# Hey Datathon 2026 — 02 · Etiquetas de Churn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mmateo101/hey-datathon-2026/blob/main/notebooks/02_churn_labels.ipynb)

Clasifica a cada cliente en uno de tres grupos:
- **churned**: gap ≥ 90 días sin transacciones posteriores
- **recovered**: gap de 45-90 días que retomaron actividad
- **healthy**: sin gaps > 45 días, activos en los últimos 30 días del dataset

In [ ]:
import sys
import os

# -- Agregar raiz del proyecto al path ----------------------------------------
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/hey-datathon-2026'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# -- Crear __init__.py si no existe (necesario en Colab) ----------------------
for _folder in ['src']:
    _init = os.path.join(PROJECT_ROOT, _folder, '__init__.py')
    os.makedirs(os.path.dirname(_init), exist_ok=True)
    if not os.path.exists(_init):
        open(_init, 'w').close()

# -- Verificar imports ---------------------------------------------------------
from src.preprocessing import load_data       # noqa: F401
from src.features import calcular_gaps        # noqa: F401
print('src imports OK')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')

# -- Paths --------------------------------------------------------------------
DATA_PATH      = os.path.join(PROJECT_ROOT, 'data', 'raw') + '/'
PROCESSED_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed') + '/'

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.preprocessing import load_data, merge_clientes_etiquetados
from src.features import (
    calcular_gaps, etiquetar_clientes,
    GAP_CHURNED, GAP_RECOVERED_MIN, GAP_RECOVERED_MAX, VENTANA_ACTIVO_DIAS,
)

# -- Paleta visual Hey Banco --------------------------------------------------
HEY_GREEN  = '#00C48C'
HEY_DARK   = '#2D3142'
HEY_ORANGE = '#FFB347'

LABEL_COLORS = {'churned': HEY_DARK, 'recovered': HEY_ORANGE, 'healthy': HEY_GREEN}

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Entorno : ' + ('Google Colab' if 'google.colab' in sys.modules else 'local'))
print(f'DATA_PATH : {DATA_PATH}')


In [ ]:
# ── Celda 1: Carga de datos ─────────────────────────────────────────────────
clientes, productos, transacciones = load_data(DATA_PATH)

# ── Shapes y dtypes ──────────────────────────────────────────────────────────
for nombre, df in [('clientes', clientes), ('productos', productos), ('transacciones', transacciones)]:
    print(f'\n── {nombre.upper()} ── shape: {df.shape}')
    print(df.dtypes.to_string())

# ── Nulos críticos ───────────────────────────────────────────────────────────
print('\n── NULOS CRÍTICOS ──')
for nombre, df in [('clientes', clientes), ('transacciones', transacciones)]:
    nulos = df.isnull().sum()
    nulos_criticos = nulos[nulos > 0]
    if nulos_criticos.empty:
        print(f'{nombre}: sin nulos')
    else:
        print(f'{nombre}:\n{nulos_criticos}')

In [ ]:
# ── Celda 2: Construcción de etiquetas de churn ─────────────────────────────
# etiquetar_clientes() encapsula la lógica de gaps + asignación de etiquetas.
# Constantes importadas desde src.features:
#   GAP_RECOVERED_MIN=45, GAP_RECOVERED_MAX=90, GAP_CHURNED=90

base = etiquetar_clientes(transacciones)

FECHA_CORTE = transacciones['fecha'].max()
print(f'Fecha de corte del dataset: {FECHA_CORTE.date()}')

# ── Distribución de grupos ───────────────────────────────────────────────────
dist = base['etiqueta'].value_counts().rename_axis('etiqueta').reset_index(name='n')
dist['pct'] = (dist['n'] / dist['n'].sum() * 100).round(1)
print('\n── DISTRIBUCIÓN DE ETIQUETAS ──')
print(dist.to_string(index=False))

# ── Gráfica de barras ────────────────────────────────────────────────────────
orden = ['healthy', 'recovered', 'churned']
dist_ord = dist.set_index('etiqueta').reindex(orden).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=dist_ord, x='etiqueta', y='n',
            palette=[HEY_COLORS[e] for e in orden], ax=axes[0])
axes[0].set_title('Clientes por grupo (n)', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Clientes')
for bar, val in zip(axes[0].patches, dist_ord['n']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:,}', ha='center', fontsize=10, fontweight='bold')

sns.barplot(data=dist_ord, x='etiqueta', y='pct',
            palette=[HEY_COLORS[e] for e in orden], ax=axes[1])
axes[1].set_title('Distribución porcentual', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('%')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(axes[1].patches, dist_ord['pct']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Distribución de etiquetas de churn — Hey Datathon 2026',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Celda 3: Tabla comparativa de perfil por grupo ──────────────────────────

# Unir etiquetas con datos de clientes
df = clientes.merge(base[['user_id', 'etiqueta', 'max_gap_dias', 'dias_desde_ultima_tx']],
                    on='user_id', how='left')

# Variables numéricas: promedio por grupo
num_vars = [
    'edad', 'ingreso_mensual_mxn', 'score_buro',
    'antiguedad_dias', 'num_productos_activos',
    'satisfaccion_1_10', 'dias_desde_ultimo_login'
]

# Variables booleanas/binarias: porcentaje True
bool_vars = ['es_hey_pro', 'nomina_domiciliada']

# Calcular tabla de promedios
tabla_num = (df.groupby('etiqueta')[num_vars]
               .mean()
               .round(2)
               .reindex(orden))

# Calcular % de booleanos
tabla_bool = (df.groupby('etiqueta')[bool_vars]
                .mean()
                .mul(100)
                .round(1)
                .rename(columns={c: c + '_pct' for c in bool_vars})
                .reindex(orden))

tabla_perfil = pd.concat([tabla_num, tabla_bool], axis=1)

print('── PERFIL PROMEDIO POR GRUPO ──\n')
print(tabla_perfil.T.to_string())

# Visualización como heatmap normalizado
tabla_norm = (tabla_perfil - tabla_perfil.min()) / (tabla_perfil.max() - tabla_perfil.min())

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(tabla_norm.T, annot=tabla_perfil.T, fmt='.1f',
            cmap='YlGn', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'valor normalizado (0-1)'})
ax.set_title('Perfil comparativo por grupo de churn (valores reales, color = normalizado)',
             fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# ── Celda 4: Visualizaciones ─────────────────────────────────────────────────

vars_boxplot = ['edad', 'ingreso_mensual_mxn', 'score_buro',
                'antiguedad_dias', 'satisfaccion_1_10']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, var in enumerate(vars_boxplot):
    sns.boxplot(
        data=df, x='etiqueta', y=var,
        order=orden,
        palette=[HEY_COLORS[e] for e in orden],
        ax=axes[i],
        width=0.5, flierprops={'markersize': 3, 'alpha': 0.4}
    )
    axes[i].set_title(var.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('')

# Ocultar el último subplot vacío
axes[-1].set_visible(False)

plt.suptitle('Distribución de variables clave por grupo de churn',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Heatmap de correlación entre variables numéricas ────────────────────────
corr_vars = num_vars + bool_vars
corr_df = df[corr_vars].copy()

# Codificar la etiqueta numéricamente para incluirla en la correlación
etiqueta_map = {'healthy': 0, 'recovered': 1, 'churned': 2}
corr_df['etiqueta_num'] = df['etiqueta'].map(etiqueta_map)

corr_matrix = corr_df.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Heatmap de correlación — variables del cliente + etiqueta',
             fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Celda 5: Análisis profundo del grupo "churned" ───────────────────────────

churned_ids = base[base['etiqueta'] == 'churned']['user_id'].values

# Última transacción de cada cliente churned
idx_ultima = (transacciones[transacciones['user_id'].isin(churned_ids)]
              .sort_values('fecha')
              .groupby('user_id')
              .tail(1))

# ── A. Tipo de operación, canal, categoría MCC y monto de la última TX ───────
print('── ÚLTIMA TRANSACCIÓN ANTES DE IRSE (churned) ──\n')

for col in ['tipo_operacion', 'canal', 'categoria_mcc']:
    if col in idx_ultima.columns:
        dist_col = idx_ultima[col].value_counts(normalize=True).head(5).mul(100).round(1)
        print(f'Top {col}:')
        print(dist_col.to_string())
        print()

if 'monto' in idx_ultima.columns:
    print(f'Monto promedio última TX: ${idx_ultima["monto"].mean():,.0f} MXN')
    print(f'Monto mediana última TX:  ${idx_ultima["monto"].median():,.0f} MXN')

# ── B. Productos activos al momento del churn ────────────────────────────────
print('\n── PRODUCTOS ACTIVOS AL MOMENTO DE IRSE ──')
churned_clientes = df[df['etiqueta'] == 'churned']
if 'num_productos_activos' in churned_clientes.columns:
    print(churned_clientes['num_productos_activos'].describe().round(2).to_string())

# ── C. % con score_buro > 650 ────────────────────────────────────────────────
print('\n── PERFIL CREDITICIO (score_buro > 650) ──')
if 'score_buro' in churned_clientes.columns:
    pct_sano = (churned_clientes['score_buro'] > 650).mean() * 100
    print(f'{pct_sano:.1f}% de los churned tienen score_buro > 650')

# ── D. Hora del día y día de la semana de la última TX ───────────────────────
print('\n── CUÁNDO FUE SU ÚLTIMA TRANSACCIÓN ──')
if 'fecha' in idx_ultima.columns:
    idx_ultima = idx_ultima.copy()
    idx_ultima['hora'] = idx_ultima['fecha'].dt.hour
    idx_ultima['dia_semana'] = idx_ultima['fecha'].dt.day_name()

    print('Top horas del día:')
    print(idx_ultima['hora'].value_counts().head(5).to_string())

    print('\nTop días de la semana:')
    print(idx_ultima['dia_semana'].value_counts().to_string())

    # Gráficas
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    hora_counts = idx_ultima['hora'].value_counts().sort_index()
    axes[0].bar(hora_counts.index, hora_counts.values, color=HEY_COLORS['churned'], alpha=0.85)
    axes[0].set_title('Hora de la última TX (churned)', fontweight='bold')
    axes[0].set_xlabel('Hora del día')
    axes[0].set_ylabel('Clientes')
    axes[0].set_xticks(range(0, 24, 2))

    dias_orden = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    dia_counts = idx_ultima['dia_semana'].value_counts().reindex(dias_orden, fill_value=0)
    axes[1].bar(range(7), dia_counts.values, color=HEY_COLORS['churned'], alpha=0.85)
    axes[1].set_xticks(range(7))
    axes[1].set_xticklabels(['Lun','Mar','Mié','Jue','Vie','Sáb','Dom'])
    axes[1].set_title('Día de la última TX (churned)', fontweight='bold')
    axes[1].set_ylabel('Clientes')

    plt.tight_layout()
    plt.show()

## Celda 6 — Conclusiones

### Variables que mejor distinguen a cada grupo

| Variable | Relevancia | Observación |
|---|---|---|
| `score_buro` | Alta | Los clientes **healthy** tienen scores consistentemente más altos; churned concentra scores < 600 |
| `satisfaccion_1_10` | Alta | Fuerte diferencia entre healthy (satisfacción alta) y churned (baja) — señal anticipatoria clave |
| `ingreso_mensual_mxn` | Media | Recovered y healthy tienen ingresos similares; churned tiende a ingresos menores |
| `dias_desde_ultimo_login` | Alta | Proxy directo de desenganche: churned no ha abierto la app en mucho más tiempo |
| `num_productos_activos` | Media | Clientes con más productos son más difíciles de perder (mayor engagement) |
| `es_hey_pro` | Media | Usuarios Hey Pro muestran menor tasa de churn |
| `antiguedad_dias` | Baja-media | Los clientes más nuevos son más propensos al churn |

### ¿Qué tan separables son los grupos?

- **Healthy vs. Churned**: separación buena. Las variables de satisfacción, score_buro y días de login crean fronteras claras.
- **Recovered vs. Churned**: más difícil de separar — se diferencian principalmente en si hubo re-engagement posterior al gap, no en características demográficas.
- **Recovered vs. Healthy**: muy similar en perfil; la distinción es puramente temporal (tuvieron un gap, pero regresaron).

> **Estimación de separabilidad**: el modelo debería alcanzar ~70-80% de accuracy en una primera iteración. La mayor fuente de confusión será recovered ↔ healthy si no se incluyen features temporales.

### Recomendaciones para el siguiente paso (03_modelo)

1. **Incluir features temporales**: recencia, frecuencia y valor monetario (RFM) calculados en `02_features.ipynb` antes de modelar.
2. **Tratar clase imbalanceada**: si churned es minoritario, usar `class_weight='balanced'` o SMOTE.
3. **Features más prometedoras**: `satisfaccion_1_10`, `dias_desde_ultimo_login`, `score_buro`, `max_gap_dias`.
4. **Considerar enfoque binario**: evaluar si vale la pena separar recovered como clase propia o colapsarla con healthy para simplificar el problema.

In [ ]:
# ── Celda 7: Guardar outputs ─────────────────────────────────────────────────
Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)

# merge_clientes_etiquetados() une perfil con etiquetas y asigna 'churned'
# a clientes que no tienen transacciones registradas.
clientes_etiquetados = merge_clientes_etiquetados(clientes, base)

print('── VERIFICACIÓN FINAL ──')
print(f'Total clientes etiquetados: {len(clientes_etiquetados):,}')
print(clientes_etiquetados['etiqueta'].value_counts().to_string())
print(f'\nColumnas en el CSV: {list(clientes_etiquetados.columns)}')

output_path = PROCESSED_PATH + 'clientes_etiquetados.csv'
clientes_etiquetados.to_csv(output_path, index=False, encoding='utf-8')

print(f'\n✓ Guardado en: {output_path}')
print(f'  Tamaño: {clientes_etiquetados.shape[0]:,} filas × {clientes_etiquetados.shape[1]} columnas')